# Refactored Hybrid Model: PhoBERT + SentiWordNet Light Fusion\n\n## Design goals\n1. Keep PhoBERT close to the strong baseline by using the CLS token directly.\n2. Keep SentiWordNet as a compact auxiliary signal with interpretable cues.\n3. Compare simple concatenation against optional gated fusion.\n4. Make class weighting and selection metrics easy to change.


## 1. Setup và Import Libraries

In [1]:
import os, sys, json, time, random, copy
from pathlib import Path
from datetime import datetime
from collections import Counter

IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    drive = None

if IN_COLAB:
    drive.mount('/content/drive')

def find_project_root():
    if IN_COLAB:
        root = Path('/content/drive/MyDrive/Student-Feedback-Sentiment-Analysis')
        if root.exists():
            return root
        raise FileNotFoundError(root)
    cwd = Path.cwd().resolve()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / 'src').exists() and (candidate / 'data').exists() and (candidate / 'results').exists():
            return candidate
    raise FileNotFoundError('Project root not found')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

ValueError: mount failed

In [ ]:
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score,
    precision_score, recall_score, precision_recall_fscore_support
)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
from tqdm import tqdm
from IPython.display import display
from src.data_utils import (
    load_data, load_sentiwordnet, preprocess_vietnamese,
    extract_swn_features_extended_batch, SWN_EXTENDED_FEATURE_NAMES
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2. Configuration

In [ ]:
class Config:
    BASE_DIR = str(PROJECT_ROOT)
    DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed')
    SENTIWORDNET_FILE = os.path.join(BASE_DIR, 'data', 'sentiwordnet-dataset', 'VietSentiWordnet_Ver1.3.5.txt')
    MODEL_TYPE = 'PhoBERT_Sentiwordnet_Refactored_LightFusion'
    EXPERIMENT_TYPE = 'improvements'
    TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
    RESULTS_DIR = os.path.join(BASE_DIR, 'results', MODEL_TYPE, EXPERIMENT_TYPE, TIMESTAMP)
    MODELS_DIR = os.path.join(RESULTS_DIR, 'models')
    SUMMARIES_DIR = os.path.join(RESULTS_DIR, 'summaries')
    VISUALIZATIONS_DIR = os.path.join(RESULTS_DIR, 'visualizations')
    ARTIFACTS_DIR = os.path.join(RESULTS_DIR, 'artifacts')
    MODEL_NAME = 'vinai/phobert-base'
    LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    NUM_CLASSES = 3
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MAX_LENGTH = 256
    BATCH_SIZE = 32
    EPOCHS = 15
    EARLY_STOP_PATIENCE = 2
    WARMUP_RATIO = 0.1
    GRADIENT_CLIP = 1.0
    WEIGHT_DECAY = 0.01
    DROPOUT = 0.3
    PHOBERT_LR_FROZEN = 0.0
    PHOBERT_LR_PARTIAL = 1e-5
    PHOBERT_LR_FULL = 2e-5
    HEAD_LR = 5e-5
    FREEZE_EPOCHS = 1
    PARTIAL_UNFREEZE_EPOCH = 2
    FULL_UNFREEZE_EPOCH = 4
    PARTIAL_UNFREEZE_LAST_N = 4
    FUSION_MODE = 'concat'
    SWN_PROJ_DIM = 64
    CLASSIFIER_HIDDEN_DIM = 256
    SELECTION_METRIC = 'f1_macro'
    CLASS_WEIGHT_MODE = 'sqrt_balanced'

config = Config()
for path in [config.RESULTS_DIR, config.MODELS_DIR, config.SUMMARIES_DIR, config.VISUALIZATIONS_DIR, config.ARTIFACTS_DIR]:
    os.makedirs(path, exist_ok=True)

assert config.FUSION_MODE in {'concat', 'gated'}
assert config.SELECTION_METRIC in {'f1_macro', 'f1_weighted'}
assert config.CLASS_WEIGHT_MODE in {'none', 'sqrt_balanced', 'log1p_balanced', 'balanced'}

print('=' * 60)
print('REFACTORED CONFIGURATION')
print('=' * 60)
print(f'Model: {config.MODEL_NAME}')
print(f'Fusion Mode: {config.FUSION_MODE}')
print(f'Selection Metric: {config.SELECTION_METRIC}')
print(f'Class Weight Mode: {config.CLASS_WEIGHT_MODE}')
print(f'Batch Size: {config.BATCH_SIZE}')
print(f'Epochs: {config.EPOCHS}')
print(f'Device: {config.DEVICE}')
print(f'Results Dir: {config.RESULTS_DIR}')


## 3. Load Data

In [ ]:
train_texts, train_labels = load_data(config.DATA_DIR, 'train')
val_texts, val_labels = load_data(config.DATA_DIR, 'validation')
test_texts, test_labels = load_data(config.DATA_DIR, 'test')

print('Label distribution:')
for split_name, labels in [('Train', train_labels), ('Validation', val_labels), ('Test', test_labels)]:
    counter = Counter(labels)
    print(f'{split_name}: total={len(labels)}')
    for idx, name in config.LABEL_MAP.items():
        count = counter.get(idx, 0)
        print(f'  {name}: {count} ({count/len(labels)*100:.1f}%)')

## 4. Load SentiWordNet & Build Compact Auxiliary Features


In [ ]:
INTENSIFIER_WORDS = {'rất', 'quá', 'khá', 'hơi', 'siêu', 'cực', 'thật', 'thực_sự', 'vô_cùng'}
EXTRA_SENTIMENT_CUE_NAMES = ['exclamation_count', 'question_count', 'has_exclamation', 'has_question', 'intensifier_count', 'intensifier_ratio']
AUX_FEATURE_NAMES = SWN_EXTENDED_FEATURE_NAMES + EXTRA_SENTIMENT_CUE_NAMES


def extract_additional_sentiment_cues(text):
    words = preprocess_vietnamese(text).split()
    token_count = max(len(words), 1)
    exclamation_count = text.count('!')
    question_count = text.count('?')
    intensifier_count = sum(1 for word in words if word in INTENSIFIER_WORDS)
    return [
        exclamation_count,
        question_count,
        float(exclamation_count > 0),
        float(question_count > 0),
        intensifier_count,
        intensifier_count / token_count,
    ]


def build_swn_aux_features(texts, word_to_scores):
    base_features = extract_swn_features_extended_batch(texts, word_to_scores).astype(np.float32)
    extra_cues = np.array([extract_additional_sentiment_cues(text) for text in texts], dtype=np.float32)
    return np.concatenate([base_features, extra_cues], axis=1)


print('Loading VietSentiWordNet lexicon...')
word_to_scores = load_sentiwordnet(config.SENTIWORDNET_FILE)
print(f'Loaded {len(word_to_scores)} words')
print('\nExtracting compact SentiWordNet + cue features...')
train_swn = build_swn_aux_features(train_texts, word_to_scores)
val_swn = build_swn_aux_features(val_texts, word_to_scores)
test_swn = build_swn_aux_features(test_texts, word_to_scores)
print(f'Auxiliary feature shape: Train={train_swn.shape}, Val={val_swn.shape}, Test={test_swn.shape}')
print('Key cues included: pos/neg counts, ratios, negation signals, punctuation cues, and intensifier indicators.')
swn_scaler = StandardScaler()
train_swn_scaled = swn_scaler.fit_transform(train_swn).astype(np.float32)
val_swn_scaled = swn_scaler.transform(val_swn).astype(np.float32)
test_swn_scaled = swn_scaler.transform(test_swn).astype(np.float32)
joblib.dump(swn_scaler, os.path.join(config.ARTIFACTS_DIR, 'swn_scaler.pkl'))
with open(os.path.join(config.ARTIFACTS_DIR, 'swn_feature_names.json'), 'w', encoding='utf-8') as f:
    json.dump(AUX_FEATURE_NAMES, f, ensure_ascii=False, indent=2)


## 5. Dataset & Model Definition


In [ ]:
class HybridDataset(Dataset):
    """Dataset that returns PhoBERT inputs plus compact sentiment lexicon features."""
    def __init__(self, texts, swn_features, labels, tokenizer, max_length):
        self.texts = texts
        self.swn_features = swn_features
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'swn_features': torch.tensor(self.swn_features[idx], dtype=torch.float),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [ ]:
class PhoBERTSentiWordNetLightHybrid(nn.Module):
    """PhoBERT-first hybrid with a compact interpretable sentiment-feature branch."""
    def __init__(self, model_name, num_classes, swn_dim, fusion_mode='concat', dropout=0.3):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        self.fusion_mode = fusion_mode
        phobert_dim = self.phobert.config.hidden_size
        self.phobert_dropout = nn.Dropout(dropout)
        self.swn_projection = nn.Sequential(
            nn.Linear(swn_dim, config.SWN_PROJ_DIM),
            nn.LayerNorm(config.SWN_PROJ_DIM),
            nn.SiLU(),
            nn.Dropout(dropout)
        )
        combined_dim = phobert_dim + config.SWN_PROJ_DIM
        if fusion_mode == 'gated':
            self.aux_gate = nn.Linear(combined_dim, config.SWN_PROJ_DIM)
            nn.init.constant_(self.aux_gate.bias, -1.0)
        else:
            self.aux_gate = None
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(combined_dim, config.CLASSIFIER_HIDDEN_DIM),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(config.CLASSIFIER_HIDDEN_DIM, num_classes)
        )

    def get_head_modules(self):
        modules = [self.swn_projection, self.classifier]
        if self.aux_gate is not None:
            modules.append(self.aux_gate)
        return modules

    def forward(self, input_ids, attention_mask, swn_features):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        phobert_cls = self.phobert_dropout(outputs.last_hidden_state[:, 0, :])
        swn_projected = self.swn_projection(swn_features)
        if self.aux_gate is not None:
            gate = torch.sigmoid(self.aux_gate(torch.cat([phobert_cls, swn_projected], dim=1)))
            swn_projected = swn_projected * gate
        return self.classifier(torch.cat([phobert_cls, swn_projected], dim=1))


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
model = PhoBERTSentiWordNetLightHybrid(
    model_name=config.MODEL_NAME,
    num_classes=config.NUM_CLASSES,
    swn_dim=train_swn_scaled.shape[1],
    fusion_mode=config.FUSION_MODE,
    dropout=config.DROPOUT
).to(config.DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Auxiliary feature dim: {train_swn_scaled.shape[1]}')
print(f'Total parameters: {total_params:,}')
print(f'Initial trainable parameters: {count_trainable_params(model):,}')


## 6. Create DataLoaders


In [ ]:
train_dataset = HybridDataset(train_texts, train_swn_scaled, train_labels, tokenizer, config.MAX_LENGTH)
val_dataset = HybridDataset(val_texts, val_swn_scaled, val_labels, tokenizer, config.MAX_LENGTH)
test_dataset = HybridDataset(test_texts, test_swn_scaled, test_labels, tokenizer, config.MAX_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')


## 7. Training Setup with Gradual Unfreezing


In [ ]:
def build_class_weight_tensor(labels, mode, device):
    classes = np.array(sorted(set(labels)))
    balanced = compute_class_weight(class_weight='balanced', classes=classes, y=np.array(labels)).astype(np.float32)
    if mode == 'none':
        return None, balanced, None
    if mode == 'sqrt_balanced':
        applied = np.sqrt(balanced)
    elif mode == 'log1p_balanced':
        applied = np.log1p(balanced)
    elif mode == 'balanced':
        applied = balanced
    else:
        raise ValueError(f'Unsupported class weight mode: {mode}')
    return torch.tensor(applied, dtype=torch.float, device=device), balanced, applied


class_weights, raw_class_weights, applied_class_weights = build_class_weight_tensor(
    train_labels, config.CLASS_WEIGHT_MODE, config.DEVICE
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print(f'Raw balanced weights: {raw_class_weights.tolist()}')
if applied_class_weights is None:
    print('Class weight mode: none')
else:
    print(f'Applied class weights ({config.CLASS_WEIGHT_MODE}): {applied_class_weights.tolist()}')


def get_training_stage(epoch_number):
    if epoch_number <= config.FREEZE_EPOCHS:
        return 'frozen'
    if epoch_number < config.FULL_UNFREEZE_EPOCH:
        return 'partial'
    return 'full'


def set_phobert_trainable_layers(model, stage):
    for param in model.phobert.parameters():
        param.requires_grad = False
    if stage == 'partial':
        for layer in model.phobert.encoder.layer[-config.PARTIAL_UNFREEZE_LAST_N:]:
            for param in layer.parameters():
                param.requires_grad = True
    elif stage == 'full':
        for param in model.phobert.parameters():
            param.requires_grad = True
    for module in model.get_head_modules():
        for param in module.parameters():
            param.requires_grad = True


def build_optimizer_and_scheduler(model, stage, epoch_number):
    phobert_lr = {
        'frozen': config.PHOBERT_LR_FROZEN,
        'partial': config.PHOBERT_LR_PARTIAL,
        'full': config.PHOBERT_LR_FULL
    }[stage]
    groups = []
    phobert_params = [p for p in model.phobert.parameters() if p.requires_grad]
    if phobert_params:
        groups.append({'params': phobert_params, 'lr': phobert_lr})
    head_params = [p for module in model.get_head_modules() for p in module.parameters() if p.requires_grad]
    groups.append({'params': head_params, 'lr': config.HEAD_LR})
    optimizer = torch.optim.AdamW(groups, weight_decay=config.WEIGHT_DECAY)
    remaining_epochs = config.EPOCHS - epoch_number + 1
    total_steps = max(1, remaining_epochs * len(train_loader))
    warmup_steps = int(total_steps * config.WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    return optimizer, scheduler, phobert_lr


def get_selection_score(metrics):
    return metrics[config.SELECTION_METRIC]


set_phobert_trainable_layers(model, 'frozen')
print(f'Trainable params after freeze: {count_trainable_params(model):,}')


## 8. Training Loop


In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for batch in tqdm(dataloader, desc='Training', leave=False):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        swn_features = batch['swn_features'].to(device)
        labels = batch['label'].to(device)
        logits = model(input_ids, attention_mask, swn_features)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_weighted': f1_score(all_labels, all_preds, average='weighted', zero_division=0),
    }


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating', leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            swn_features = batch['swn_features'].to(device)
            labels = batch['label'].to(device)
            logits = model(input_ids, attention_mask, swn_features)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
    precision_pc, recall_pc, f1_pc, support_pc = precision_recall_fscore_support(
        all_labels, all_preds, labels=[0, 1, 2], zero_division=0
    )
    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision_macro': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall_macro': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'precision_weighted': precision_score(all_labels, all_preds, average='weighted', zero_division=0),
        'recall_weighted': recall_score(all_labels, all_preds, average='weighted', zero_division=0),
        'f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_weighted': f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        'precision_per_class': precision_pc.tolist(),
        'recall_per_class': recall_pc.tolist(),
        'f1_per_class': f1_pc.tolist(),
        'support_per_class': support_pc.tolist(),
        'confusion_matrix': confusion_matrix(all_labels, all_preds, labels=[0, 1, 2]).tolist(),
        'classification_report_text': classification_report(all_labels, all_preds, labels=[0, 1, 2], target_names=list(config.LABEL_MAP.values()), zero_division=0),
        'y_pred': list(map(int, all_preds)),
        'y_true': list(map(int, all_labels)),
    }

def print_split_metrics(split_name, metrics):
    print(f'{split_name} Metrics:')
    print(f'  Accuracy: {metrics["accuracy"]:.4f}')
    print(f'  Precision Macro: {metrics["precision_macro"]:.4f}')
    print(f'  Recall Macro: {metrics["recall_macro"]:.4f}')
    print(f'  F1 Macro: {metrics["f1_macro"]:.4f}')
    print(f'  Precision Weighted: {metrics["precision_weighted"]:.4f}')
    print(f'  Recall Weighted: {metrics["recall_weighted"]:.4f}')
    print(f'  F1 Weighted: {metrics["f1_weighted"]:.4f}')
    print('Classification Report:')
    print(metrics['classification_report_text'])
    print('Confusion Matrix:')
    print(np.array(metrics['confusion_matrix']))


In [ ]:
history = {
    'epoch': [], 'stage': [],
    'train_loss': [], 'train_acc': [], 'train_f1_macro': [], 'train_f1_weighted': [],
    'val_loss': [], 'val_acc': [], 'val_f1_macro': [], 'val_f1_weighted': [],
    'val_f1_neutral': [], 'val_selection_score': [], 'phobert_lr': [], 'trainable_params': []
}

best_val_score = -1.0
best_val_f1 = -1.0
best_epoch = 0
best_stage = None
best_model_state = None
patience_counter = 0
current_stage = None

print('=' * 70)
print('START TRAINING WITH GRADUAL UNFREEZING')
print('=' * 70)

for epoch in range(1, config.EPOCHS + 1):
    stage = get_training_stage(epoch)
    if stage != current_stage:
        current_stage = stage
        set_phobert_trainable_layers(model, stage)
        optimizer, scheduler, phobert_lr = build_optimizer_and_scheduler(model, stage, epoch)
        print(f'\n[Stage Switch] epoch={epoch} stage={stage} phobert_lr={phobert_lr} trainable_params={count_trainable_params(model):,}')

    print(f'\nEpoch {epoch}/{config.EPOCHS}')
    train_metrics = train_epoch(model, train_loader, criterion, optimizer, scheduler, config.DEVICE)
    val_metrics = evaluate(model, val_loader, criterion, config.DEVICE)
    selection_score = get_selection_score(val_metrics)

    history['epoch'].append(epoch)
    history['stage'].append(stage)
    history['train_loss'].append(train_metrics['loss'])
    history['train_acc'].append(train_metrics['accuracy'])
    history['train_f1_macro'].append(train_metrics['f1_macro'])
    history['train_f1_weighted'].append(train_metrics['f1_weighted'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['val_f1_macro'].append(val_metrics['f1_macro'])
    history['val_f1_weighted'].append(val_metrics['f1_weighted'])
    history['val_f1_neutral'].append(val_metrics['f1_per_class'][1])
    history['val_selection_score'].append(selection_score)
    history['phobert_lr'].append(phobert_lr)
    history['trainable_params'].append(count_trainable_params(model))

    print(f"Train  - loss={train_metrics['loss']:.4f} acc={train_metrics['accuracy']:.4f} f1_macro={train_metrics['f1_macro']:.4f} f1_weighted={train_metrics['f1_weighted']:.4f}")
    print(f"Val    - loss={val_metrics['loss']:.4f} acc={val_metrics['accuracy']:.4f} f1_macro={val_metrics['f1_macro']:.4f} f1_weighted={val_metrics['f1_weighted']:.4f} f1_neutral={val_metrics['f1_per_class'][1]:.4f} selected={selection_score:.4f}")

    if selection_score > best_val_score:
        best_val_score = selection_score
        best_val_f1 = val_metrics['f1_macro']
        best_epoch = epoch
        best_stage = stage
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        torch.save(best_model_state, os.path.join(config.MODELS_DIR, 'best_model.pt'))
        print(f'  -> New best model saved ({config.SELECTION_METRIC}={best_val_score:.4f})')
    else:
        patience_counter += 1
        print(f'  -> No improvement, patience={patience_counter}/{config.EARLY_STOP_PATIENCE}')
        if patience_counter >= config.EARLY_STOP_PATIENCE:
            print(f'  -> Early stopping at epoch {epoch}')
            break

assert best_model_state is not None
model.load_state_dict(best_model_state)
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(config.SUMMARIES_DIR, 'training_history.csv'), index=False)
print(f'\nBest epoch: {best_epoch}, best stage: {best_stage}, best {config.SELECTION_METRIC}: {best_val_score:.4f}, best val macro F1: {best_val_f1:.4f}')


## 9. Evaluation on Validation and Test Sets


In [ ]:
val_results = evaluate(model, val_loader, criterion, config.DEVICE)
test_results = evaluate(model, test_loader, criterion, config.DEVICE)
print_split_metrics('Validation', val_results)
print()
print_split_metrics('Test', test_results)


## 10. Save Results


In [ ]:
summary_df = pd.DataFrame([
    {'Split': 'Validation', 'Best_Epoch': best_epoch, 'Best_Stage': best_stage, 'Selection_Metric': config.SELECTION_METRIC, 'Selection_Score': best_val_score, 'Accuracy': val_results['accuracy'], 'Precision_Macro': val_results['precision_macro'], 'Recall_Macro': val_results['recall_macro'], 'F1_Macro': val_results['f1_macro'], 'F1_Weighted': val_results['f1_weighted'], 'F1_Negative': val_results['f1_per_class'][0], 'F1_Neutral': val_results['f1_per_class'][1], 'F1_Positive': val_results['f1_per_class'][2]},
    {'Split': 'Test', 'Best_Epoch': best_epoch, 'Best_Stage': best_stage, 'Selection_Metric': config.SELECTION_METRIC, 'Selection_Score': best_val_score, 'Accuracy': test_results['accuracy'], 'Precision_Macro': test_results['precision_macro'], 'Recall_Macro': test_results['recall_macro'], 'F1_Macro': test_results['f1_macro'], 'F1_Weighted': test_results['f1_weighted'], 'F1_Negative': test_results['f1_per_class'][0], 'F1_Neutral': test_results['f1_per_class'][1], 'F1_Positive': test_results['f1_per_class'][2]},
])
display(summary_df)
summary_df.to_csv(os.path.join(config.SUMMARIES_DIR, 'summary.csv'), index=False)

experiment_summary = {
    'model_type': config.MODEL_TYPE,
    'selection_metric': config.SELECTION_METRIC,
    'best_selection_score': best_val_score,
    'best_val_f1_macro': best_val_f1,
    'class_weight_mode': config.CLASS_WEIGHT_MODE,
    'class_weights_raw': raw_class_weights.tolist(),
    'class_weights_applied': None if applied_class_weights is None else applied_class_weights.tolist(),
    'fusion_mode': config.FUSION_MODE,
    'aux_feature_count': len(AUX_FEATURE_NAMES),
    'aux_feature_names': AUX_FEATURE_NAMES,
    'validation': val_results,
    'test': test_results,
}
with open(os.path.join(config.SUMMARIES_DIR, 'experiment_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(experiment_summary, f, ensure_ascii=False, indent=2)

with open(os.path.join(config.SUMMARIES_DIR, 'training_results.txt'), 'w', encoding='utf-8') as f:
    f.write('=' * 60 + '\n')
    f.write('TRAINING RESULTS - PhoBERT + SentiWordNet Refactored Light Fusion\n')
    f.write('=' * 60 + '\n')
    f.write(f'Best Epoch: {best_epoch}\n')
    f.write(f'Best Stage: {best_stage}\n')
    f.write(f'Selection Metric: {config.SELECTION_METRIC}\n')
    f.write(f'Best Selection Score: {best_val_score:.4f}\n')
    f.write(f'Fusion Mode: {config.FUSION_MODE}\n')
    f.write(f'Class Weight Mode: {config.CLASS_WEIGHT_MODE}\n')
    f.write(f'Test Accuracy: {test_results["accuracy"]:.4f}\n')
    f.write(f'Test Precision Macro: {test_results["precision_macro"]:.4f}\n')
    f.write(f'Test Recall Macro: {test_results["recall_macro"]:.4f}\n')
    f.write(f'Test F1 Macro: {test_results["f1_macro"]:.4f}\n')
    f.write(f'Test F1 Weighted: {test_results["f1_weighted"]:.4f}\n')

print(f'Saved outputs to: {config.RESULTS_DIR}')


## 11. Visualization


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history_df['epoch'], history_df['train_loss'], marker='o', label='Train')
axes[0].plot(history_df['epoch'], history_df['val_loss'], marker='o', label='Val')
axes[0].axvline(best_epoch, color='red', linestyle='--', label=f'Best ({best_epoch})')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_f1_macro'], marker='o', label='Train Macro F1')
axes[1].plot(history_df['epoch'], history_df['val_f1_macro'], marker='o', label='Val Macro F1')
axes[1].plot(history_df['epoch'], history_df['val_f1_weighted'], marker='o', label='Val Weighted F1')
axes[1].axvline(best_epoch, color='red', linestyle='--')
axes[1].set_title('F1 Comparison')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(history_df['epoch'], history_df['val_f1_neutral'], marker='o', color='orange', label='Neutral F1')
axes[2].plot(history_df['epoch'], history_df['val_selection_score'], marker='o', color='green', label=f'Selected: {config.SELECTION_METRIC}')
axes[2].axvline(best_epoch, color='red', linestyle='--')
axes[2].set_title('Validation Monitoring')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

cm = np.array(test_results['confusion_matrix'])
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm.astype(float), row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=list(config.LABEL_MAP.values()), yticklabels=list(config.LABEL_MAP.values()), ax=axes[0])
axes[0].set_title('Confusion Matrix (Count)')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=list(config.LABEL_MAP.values()), yticklabels=list(config.LABEL_MAP.values()), ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)')
plt.tight_layout()
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()


## 12. Error Analysis for Neutral Class


In [ ]:
# Find misclassified Neutral samples
neutral_indices = [i for i, label in enumerate(test_results['y_true']) if label == 1]
misclassified_neutral = [
    (i, test_results['y_pred'][i], test_texts[i])
    for i in neutral_indices
    if test_results['y_pred'][i] != 1
]

print(f'Total Neutral samples: {len(neutral_indices)}')
print(f'Misclassified Neutral: {len(misclassified_neutral)}')
print(f'Neutral accuracy: {1 - len(misclassified_neutral)/len(neutral_indices):.2%}')

print('\n--- Sample Misclassified Neutral Feedback ---')
for idx, pred, text in misclassified_neutral[:10]:
    pred_label = config.LABEL_MAP[pred]
    print(f'  Predicted: {pred_label} | Text: "{text[:80]}..."' if len(text) > 80 else f'  Predicted: {pred_label} | Text: "{text}"')

# Count which classes Neutral is confused with
confusion_targets = Counter([pred for _, pred, _ in misclassified_neutral])
print(f'\nNeutral confusion targets:')
for cls, count in sorted(confusion_targets.items()):
    print(f'  {config.LABEL_MAP[cls]}: {count}')

## What was changed

- Replaced positional encoding and mean pooling with the native PhoBERT CLS representation: `outputs.last_hidden_state[:, 0, :]`.
- Replaced the heavier hybrid heads with a lightweight auxiliary projection plus `concat` or optional `gated` fusion.
- Made class-weight handling configurable with `none`, `sqrt_balanced`, `log1p_balanced`, and `balanced`.
- Made best-model selection configurable between `f1_macro` and `f1_weighted`.
- Kept the evaluation, confusion matrix, per-class report, and neutral-class analysis sections runnable.


## Why these changes were made

- PhoBERT is the strongest semantic encoder in these notebooks, so the hybrid branch should support it rather than distort it.
- A small auxiliary branch keeps TF-IDF or SentiWordNet useful without letting those features dominate the model.
- Softer class weights are safer for the rare Neutral class and easier to compare in ablations.
- Clearer metric selection makes experiments more reproducible and easier to interpret.


## 13. Final Summary


In [ ]:
print('=' * 70)
print('PHOBERT + SENTIWORDNET REFACTORED LIGHT FUSION - FINAL SUMMARY')
print('=' * 70)
print('\nArchitecture Choices:')
print('  1. PhoBERT sentence representation uses outputs.last_hidden_state[:, 0, :]')
print('  2. No external positional encoding and no mean pooling')
print(f'  3. Lightweight SentiWordNet auxiliary branch with fusion mode: {config.FUSION_MODE}')
print(f'  4. Small classifier head with hidden dim {config.CLASSIFIER_HIDDEN_DIM}')
print(f'  5. Class weight mode: {config.CLASS_WEIGHT_MODE}')
print(f'  6. Auxiliary feature count: {len(AUX_FEATURE_NAMES)}')
print(f'\nBest epoch: {best_epoch}')
print(f'Best stage: {best_stage}')
print(f'Best {config.SELECTION_METRIC}: {best_val_score:.4f}')
print(f'Best val macro F1: {best_val_f1:.4f}')
print('\nTest Results:')
print(f'  Accuracy: {test_results["accuracy"]:.4f}')
print(f'  Precision Macro: {test_results["precision_macro"]:.4f}')
print(f'  Recall Macro: {test_results["recall_macro"]:.4f}')
print(f'  F1 Macro: {test_results["f1_macro"]:.4f}')
print(f'  F1 Weighted: {test_results["f1_weighted"]:.4f}')
print('\nConfusion Matrix:')
print(np.array(test_results['confusion_matrix']))
print('\n' + '=' * 70)


## Recommended next experiments

- Compare `FUSION_MODE = 'concat'` against `FUSION_MODE = 'gated'`.
- Compare `CLASS_WEIGHT_MODE = 'none'`, `'sqrt_balanced'`, and `'balanced'`.
- Ablate the punctuation and intensifier cues to see whether they add stable value.
- Try smaller auxiliary projections such as `32` and `48`.
- Compare `SELECTION_METRIC = 'f1_macro'` versus `'f1_weighted'`.
